In [1]:
# Importing necessary modules at the top of the file
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
from selenium.common.exceptions import ElementClickInterceptedException, SessionNotCreatedException
from datetime import datetime
import os  # Added for directory operations

# Function to set up the WebDriver (robust version with multiple approaches)
def setup_driver():
    options = Options()
    options.add_argument("start-maximized")
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("window-size=1920,1080")
    prefs = {"profile.managed_default_content_settings.images": 2}
    options.add_experimental_option("prefs", prefs)
    
    try:
        # First attempt: Try direct Chrome initialization (works in newer selenium)
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        driver = webdriver.Chrome(options=options)
    except Exception as e:
        print(f"First attempt failed: {e}")
        try:
            # Second attempt: Use ChromeDriverManager with latest driver
            driver = webdriver.Chrome(
                service=Service(ChromeDriverManager(version="latest").install()), 
                options=options
            )
        except Exception as e2:
            print(f"Second attempt failed: {e2}")
            try:
                # Third attempt: Use Chrome service directly
                from selenium.webdriver.chrome.service import Service as ChromeService
                driver = webdriver.Chrome(service=ChromeService(), options=options)
            except Exception as e3:
                print(f"Third attempt failed: {e3}")
                # Fourth attempt: Last resort without any service
                driver = webdriver.Chrome()
    
    return driver

# Function to ensure output directory exists
def create_output_directory(base_dir='Y:\\1. Research\\Dividend History'):
    if not os.path.exists(base_dir):
        os.makedirs(base_dir)
    return base_dir

# Function to save DataFrame to CSV
def save_to_csv(df, filename, output_dir):
    filepath = os.path.join(output_dir, filename)
    df.to_csv(filepath, index=True)
    print(f"Data saved to {filepath}")

# Function to scrape data for a single fiscal year
def scrape_fiscal_year_data(driver, fy):
    try:
        # Open the target URL (only need to do this once)
        driver.get('https://www.sharesansar.com/proposed-dividend')

        # Navigate to the required section (only need to do this once)
        driver.find_element(By.XPATH, "/html/body/div[2]/div/section[2]/div[3]/div/div/div/div/div[1]/ul/li[2]/a").click()

        # Select fiscal year from dropdown
        WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.ID, 'select2-year-container'))).click()
        driver.find_element(By.CLASS_NAME, 'select2-search__field').send_keys(fy)    
        driver.find_element(By.CLASS_NAME, 'select2-search__field').send_keys(Keys.RETURN)

        # Submit the form
        driver.find_element(By.ID, "btn_pd_submit").click()

        # Wait for processing spinner to appear and disappear
        WebDriverWait(driver, 15).until(EC.visibility_of_element_located((By.ID, 'myTableFD_processing')))
        WebDriverWait(driver, 15).until(EC.invisibility_of_element_located((By.ID, 'myTableFD_processing')))
        
        # Set table length to 50
        element = driver.find_element(By.NAME, 'myTableFD_length')
        dropdown = Select(element)
        dropdown.select_by_visible_text('50')
        WebDriverWait(driver, 15).until(EC.visibility_of_element_located((By.ID, 'myTableFD_processing')))
        WebDriverWait(driver, 15).until(EC.invisibility_of_element_located((By.ID, 'myTableFD_processing')))
        
        # Scrape the first page
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')

        # Find pagination details
        pages = soup.find(id='myTableFD_paginate')
        try:
            total_pages = int(pages.find_all('a')[-2].text)
        except (IndexError, ValueError):
            total_pages = 1

        # Find the table
        table = soup.find('table', id='myTableFD')

        if not table or not table.find_all('tr'):
            return None

        headers_list = [header.text.strip() for header in table.find_all('th')]
        fy_dividends = []

        for page in range(1, total_pages + 1):
            if page > 1:
                driver.find_element(By.XPATH, '//*[@id="myTableFD_next"]').click()
                WebDriverWait(driver, 15).until(EC.visibility_of_element_located((By.ID, 'myTableFD_processing')))
                WebDriverWait(driver, 15).until(EC.invisibility_of_element_located((By.ID, 'myTableFD_processing')))
            
            html = driver.page_source
            soup = BeautifulSoup(html, 'html.parser')
            table = soup.find('table', id='myTableFD')

            output_rows = []
            for table_row in table.find_all('tr')[1:]:
                columns = table_row.find_all('td')
                output_row = [column.text.strip() for column in columns]
                output_rows.append(output_row)

            single_page_data = pd.DataFrame(output_rows)
            single_page_data.columns = headers_list
            single_page_data.set_index('S.N.', inplace=True)
            fy_dividends.append(single_page_data)
            print(f"Scraping page {page} for fiscal year {fy}")

        if fy_dividends:
            fy_data = pd.concat(fy_dividends)
            # Add fiscal year column
            fy_data['Fiscal Year'] = fy
            return fy_data
        else:
            return None
            
    except Exception as e:
        print(f"Error scraping data for fiscal year {fy}: {e}")
        return None

# Modified main scraping function to save individual and combined CSVs
def scrape_fiscal_year_dividend(fiscal_years_list):
    # Create output directory
    output_dir = create_output_directory()
    
    # Initialize WebDriver
    driver = setup_driver()
    
    all_year_dividends = []

    try:
        # Loop over fiscal years and scrape data
        for fy in fiscal_years_list:
            print(f"Scraping data for fiscal year: {fy}")
            fy_data = scrape_fiscal_year_data(driver, fy)

            if fy_data is not None and not fy_data.empty:
                # Add timestamp column
                fy_data['Timestamp'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                # Convert slashes to hyphens for filename
                filename = f"{fy.replace('/', '-')}.csv"
                save_to_csv(fy_data, filename, output_dir)
                all_year_dividends.append(fy_data)

        # Combine and save all fiscal year data
        if all_year_dividends:
            final_data = pd.concat(all_year_dividends)
            combined_filename = "all_dividend_data.csv"
            save_to_csv(final_data, combined_filename, output_dir)
            return final_data
        else:
            print("No data found for any fiscal year.")
            return None

    finally:
        # Close the browser after scraping
        driver.quit()

# Function to get fiscal years (UPDATED to use the same setup_driver function)
def sharesansar_fiscal_years():
    # Use the same robust setup_driver function we fixed earlier
    driver = setup_driver()
    
    try:
        driver.get('https://www.sharesansar.com/proposed-dividend')
        driver.find_element(By.XPATH, "/html/body/div[2]/div/section[2]/div[3]/div/div/div/div/div[1]/ul/li[2]/a").click()
        time.sleep(5)
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')
        
        fys = soup.find_all('div', class_='form-group col-md-4')[1]
        fy_list = [fy.text.strip().replace('\n\n', '') for fy in fys]
        fiscal_years_list = [year for year in fy_list[3].split() if year]
        
        # Filter years starting from 2077/2078
        start_year = '2077/2078'
        if start_year in fiscal_years_list:
            start_index = fiscal_years_list.index(start_year)
            return fiscal_years_list[:start_index + 1]
        else:
            print(f"Warning: {start_year} not found in fiscal years list")
            return []
    except Exception as e:
        print(f"Error in sharesansar_fiscal_years: {e}")
        return []
    finally:
        driver.quit()

# Example usage:
if __name__ == "__main__":
    start_time=datetime.now()
    try:
        fiscal_years = sharesansar_fiscal_years()
        if fiscal_years:
            print(f"Found fiscal years: {fiscal_years}")
            dividend_data = scrape_fiscal_year_dividend(fiscal_years)
        else:
            print("No fiscal years found. Check for errors above.")
    except Exception as e:
        print(f"An error occurred in main execution: {e}")
    end_time=datetime.now()
    duration= end_time-start_time
    print("time taken for the code:",duration)

Found fiscal years: ['2081/2082', '2080/2081', '2079/2080', '2078/2079', '2077/2078']
Scraping data for fiscal year: 2081/2082
Scraping page 1 for fiscal year 2081/2082
Data saved to Y:\1. Research\Dividend History\2081-2082.csv
Scraping data for fiscal year: 2080/2081
Scraping page 1 for fiscal year 2080/2081
Scraping page 2 for fiscal year 2080/2081
Scraping page 3 for fiscal year 2080/2081
Scraping page 4 for fiscal year 2080/2081
Data saved to Y:\1. Research\Dividend History\2080-2081.csv
Scraping data for fiscal year: 2079/2080
Scraping page 1 for fiscal year 2079/2080
Scraping page 2 for fiscal year 2079/2080
Scraping page 3 for fiscal year 2079/2080
Data saved to Y:\1. Research\Dividend History\2079-2080.csv
Scraping data for fiscal year: 2078/2079
Scraping page 1 for fiscal year 2078/2079
Scraping page 2 for fiscal year 2078/2079
Scraping page 3 for fiscal year 2078/2079
Scraping page 4 for fiscal year 2078/2079
Data saved to Y:\1. Research\Dividend History\2078-2079.csv
Scrapi